# 建立用來finetune的資料集

In [1]:
# use gemini build dataset
import os
import numpy as np
import numpy_financial as npf
import pandas as pd
import math
import csv
from datetime import datetime

from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory
from langchain_google_genai import ChatGoogleGenerativeAI

def gemini_response(model, instruction, count):
    scores = []

    for i in range(count):

        try:
            result = model.invoke(instruction)
            scores.append(int(result.content))
        except:
            pass

    if len(scores) == 0:
        score = 0
    else:
        score = sum(scores) / len(scores)

    return [date_obj.strftime('%Y%m%d'), score]  # [時間, 分數]



if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.getenv('GEMINI_API_KEY')

regenerate_count = 10
# prompt_mutate_count = 100

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)
irr_ = 0

stock_ids = ["1101", "2211", "2385", "2542", "2880", "2912", "3023", "3264", "5269", "8027"]
stock_names = ["台泥", "長榮鋼", "群光", "興富發", "華南金", "統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
# stock_ids = ["2912", "3023", "3264", "5269", "8027"]


p_mutate = '深入的公司創新分析及其對預期市場影響的評估，表明股價持續上漲，反映了投資者對公司令人振奮的長期增長前景的樂觀情緒。'

llm = ChatGoogleGenerativeAI(
    model="gemini-pro",
    temperature=1,
    safety_settings={
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    })


id = []
text = []
response = []

for idx, stock_id in enumerate(stock_ids):
    print(f'執行{stock_id}, {idx}')
    folder_path = "stock_news/" + stock_id + "/" + stock_id + "_news/"
    files = os.listdir(folder_path)
    sorted_files = sorted(files)

    # 載入買進價格
    df = pd.read_csv("price_history/" + stock_id + ".csv", dtype=str)
    price_np = df.to_numpy()
    prices = price_np.tolist()  # [時間,價格]

    signal_2 = []

    for price in prices:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(price[0], '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        

        for j, file_name in enumerate(sorted_files):
            if file_name == price[0]:
                with open(folder_path + file_name, encoding='utf-8') as f:
                    
                    general_prompt = f'請依據{p_mutate}對以下關於{stock_names[idx]}的報導，給這篇文章打一個分數，分數介於-100～100之間，-100是負向，100是最好，0則是無法判斷'

                    news = f.read()
                    # print(news)
                    prompt = general_prompt + "\n" + news
                    # gemini的回應，回傳帶有買入訊號的list
                    signal = gemini_response(llm, prompt, regenerate_count)
                    # print(price)
                    # print(signal)
                    sig = signal[1]

                    id.append(price[0])
                    text.append(news)
                    response.append(sig)
                    # [id(日期), text(新聞), llm回應]
                    # signal_2.append([price[0], news, sig])
                break
            
        # print(id)      
        # print(text)      
        # print(response)        

# todo 用pd寫入CSV文件
df = pd.DataFrame({'id': id, 'text': text, 'response': response})
# with open('dataset/id.csv', 'w', newline='', encoding='utf-8') as csvfile:
#     csv_writer = csv.writer(csvfile)
#     csv_writer.writerow(id)

# with open('dataset/text.csv', 'w', newline='', encoding='utf-8') as csvfile:
#     csv_writer = csv.writer(csvfile)
#     csv_writer.writerow(text)

# with open('dataset/response.csv', 'w', newline='', encoding='utf-8') as csvfile:
#     csv_writer = csv.writer(csvfile)
#     csv_writer.writerow(response)

/Users/yanyifu/Documents/_Coding/LLM Predict Stock/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


執行1101, 0


FileNotFoundError: [Errno 2] No such file or directory: 'stock_news/1101/1101_news/'

In [ ]:
# 改成5分標籤
import pandas as pd
import datasets
from datasets import Dataset, DatasetDict

df = pd.read_csv("dataset/signal.csv")
text = df['text'].tolist()
response = df['response'].tolist()

for i in range(len(response)):
    res_int = int(response[i])
    if res_int > 50:
        response[i] = "非常好"
    elif res_int > 0:
        response[i] = "好"
    elif res_int < -50:
        response[i] = "非常不好"
    elif res_int < -50:
        response[i] = "不好"
    else:
        response[i] = "一般"
    
print(response)
#平衡資料 移除部分標籤
for i in range(30):
    idx = response.index("一般")
    text.pop(idx)
    response.pop(idx)


# 整理上傳hugginface

In [ ]:
# 第一次需要安裝
# %pip install transformers
# %pip install huggingface_hub
# %pip install datasets

In [ ]:
import datasets
from datasets import Dataset, DatasetDict

print("檢查這兩個值是否相等")
print(len(response))
print(len(text))

train_val_splt = 0.8
trn_tst_spt = 0.9

n = int(0.8*len(response))
n2 = int(0.9*len(response))


tdf = pd.DataFrame({"text": text[:n], "response": response[:n]})
vdf = pd.DataFrame({"text": text[n:n2], "response": response[n:n2]})
ted = pd.DataFrame({"text": text[n2:], "response": response[n2:]})

tds = Dataset.from_pandas(tdf)
vds = Dataset.from_pandas(vdf)
ted = Dataset.from_pandas(ted)

ds = DatasetDict()

ds['train'] = tds
ds['validation'] = vds
ds['test'] = ted

print(ds)

# 上傳 hugginface
ds.push_to_hub(repo_id="Atomuze/stock_news_score", private=True)